# T04 — Evaluasi Akurasi dan Bias Produk Deforestasi Global di Ekosistem Hutan, Savana, dan Lahan Basah Kabupaten Merauke

Produk yang dinilai untuk **pembukaan lahan 2024–2025**: **Hansen GFC v1.13**, **JRC TMF 2025**, **RADD**, dan **Dynamic World** (penurunan peluang pohon).

| Bagian | Isi | Waktu |
|---|---|---|
| **A. Persiapan** | Peta 4 produk, strata kesepakatan (0–4 produk), ekosistem, 300 titik berstrata, potongan citra Sentinel-2 per titik | 30–45 menit, otomatis |
| **B. Interpretasi** | Aplikasi klik: Bapak menilai setiap titik dari citra 2023–2026 **tanpa melihat hasil produk** | 2–3 jam, boleh dicicil |
| **C. Analisis** | Akurasi pengguna/produser, F1, luas pembukaan + selang kepercayaan 95% (estimator berstrata), bias per ekosistem, kesepakatan antarproduk, konsistensi interpreter | 5–10 menit, otomatis |

**Penyimpanan:** semua berkas disimpan di Google Drive `MyDrive/T04_interpretasi/`, sehingga Colab boleh ditutup lalu dilanjutkan. Saat kembali: jalankan sel 0–2, lalu langsung ke bagian yang ingin dilanjutkan.

In [ ]:
# 0. Instalasi
!pip install -q earthengine-api scikit-learn pandas matplotlib seaborn openpyxl requests ipywidgets

In [ ]:
# 1. Earth Engine + Google Drive
import ee
GEE_PROJECT = 'riset-psn-pps'
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
print('Earth Engine siap:', ee.String('OK').getInfo())
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/T04_interpretasi'
except Exception:
    BASE = './T04_interpretasi'
os.makedirs(BASE, exist_ok=True); print('Folder kerja:', BASE)

## Konfigurasi

In [ ]:
import json, time, math, random, zipfile, io
import numpy as np, pandas as pd
TOPIC = 'T04'; SEED = 42
CHIPS = os.path.join(BASE, 'chips'); os.makedirs(CHIPS, exist_ok=True)
F_POINTS = os.path.join(BASE, 'T04_titik_sampel.csv'); F_STRATA = os.path.join(BASE, 'T04_bobot_strata.json')
F_LABELS = os.path.join(BASE, 'T04_label_interpretasi.csv'); F_RELAB = os.path.join(BASE, 'T04_label_ulang.csv')
F_WALL = os.path.join(BASE, 'T04_luas_produk.json'); F_AGREE = os.path.join(BASE, 'T04_sampel_kesepakatan.csv')
OUT = os.path.join(BASE, 'T04_hasil'); DIR_TAB, DIR_FIG, DIR_DATA = [os.path.join(OUT, d) for d in ('03_tabel', '02_gambar', '07_dataset')]
for d in (DIR_TAB, DIR_FIG, DIR_DATA): os.makedirs(d, exist_ok=True)
ALLOC = {0: 60, 1: 80, 2: 60, 3: 50, 4: 50}            # jumlah titik per strata (jumlah produk yang menandai pembukaan)
PRODUCTS = ['Hansen', 'TMF', 'RADD', 'DW']
ECO = {1: 'Hutan rapat', 2: 'Hutan terbuka / savana berkayu', 3: 'Savana / rumput', 4: 'Lahan basah'}
ECO_EN = {1: 'Dense forest', 2: 'Open forest / wooded savanna', 3: 'Savanna / grassland', 4: 'Wetland'}
TR = {'ID': {'Hansen': 'Hansen GFC', 'TMF': 'JRC TMF', 'RADD': 'RADD', 'DW': 'Dynamic World'}, 'EN': {'Hansen': 'Hansen GFC', 'TMF': 'JRC TMF', 'RADD': 'RADD', 'DW': 'Dynamic World'}}
COLS_EN = {'produk': 'product', 'ekosistem': 'ecosystem', 'strata': 'stratum', 'jumlah_titik': 'n_points', 'luas_ha': 'area_ha',
           'bobot': 'weight', 'akurasi_keseluruhan': 'overall_accuracy', 'akurasi_pengguna': 'users_accuracy', 'akurasi_produser': 'producers_accuracy',
           'batas_bawah': 'ci_low', 'batas_atas': 'ci_high', 'luas_peta_ha': 'mapped_area_ha', 'luas_estimasi_ha': 'estimated_area_ha',
           'kelas_interpretasi': 'interpreted_class', 'jumlah': 'count', 'persen': 'percent', 'produk_A': 'product_A', 'produk_B': 'product_B',
           'kesepakatan': 'agreement', 'n_titik_ekosistem': 'n_points_ecosystem', 'n_referensi_dibuka': 'n_reference_cleared'}
VAL_EN = {**{v: ECO_EN[k] for k, v in ECO.items()}, 'akurasi_keseluruhan': 'overall_accuracy', 'akurasi_pengguna': 'users_accuracy',
          'akurasi_produser': 'producers_accuracy', 'Referensi (interpretasi)': 'Reference (interpretation)',
          '1 Dibuka 2024–2025': '1 Cleared 2024–2025', '0 Tidak berubah': '0 No change', '2 Sudah terbuka sejak 2023': '2 Already open in 2023',
          '3 Berubah alami': '3 Natural change', '9 Tidak dapat ditentukan': '9 Undeterminable'}
COLS_EN.update({'metrik': 'metric', 'nilai': 'value', 'PA_bawah': 'PA_ci_low', 'PA_atas': 'PA_ci_high'})
def save_tab(df, name_id, name_en):
    d_en = df.copy()
    for c in d_en.columns:
        if not pd.api.types.is_numeric_dtype(d_en[c]): d_en[c] = d_en[c].map(lambda v: VAL_EN.get(v, v))
    d_en.columns = [c.replace('luas_peta_', 'mapped_area_') if c.startswith('luas_peta_') else c for c in d_en.columns]
    df.to_csv(os.path.join(DIR_TAB, f'T04_tabel_{name_id}_ID.csv'), index=False)
    d_en.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'T04_table_{name_en}_EN.csv'), index=False)

# Bagian A — Persiapan otomatis
Wilayah: seluruh Kabupaten Merauke, tanpa air permanen (JRC GSW ≥ 50%) dan lahan terbangun (ESA WorldCover).

| Produk | Definisi "pembukaan 2024–2025" |
|---|---|
| Hansen GFC v1.13 | `lossyear` = 2024 atau 2025 |
| JRC TMF 2025 | Des-2023 = hutan tak terganggu/terdegradasi/regrowth **dan** Des-2025 = deforestasi atau degradasi baru |
| RADD | alert dengan tanggal 2024-01-01 s.d. 2025-12-31 |
| Dynamic World | rata-rata peluang `trees` 2023 ≥ 0,5 **dan** Jun–Des 2025 < 0,3 |

**Strata** = jumlah produk yang menandai pembukaan (0–4). **Ekosistem**: hutan rapat (tutupan pohon 2000 ≥ 50%), hutan terbuka/savana berkayu (10–50%), savana/rumput (< 10%), lahan basah (ESA WorldCover lahan basah/mangrove; prioritas).

In [ ]:
kab = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Merauke')); AOI = kab.geometry()
gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13'); lossyear = gfc.select('lossyear').unmask(0); tc = gfc.select('treecover2000')
f_hansen = lossyear.gte(24).And(lossyear.lte(25))
tmf = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
t23, t25 = tmf.select('Dec2023'), tmf.select('Dec2025')
f_tmf = (t23.eq(1).Or(t23.eq(2)).Or(t23.eq(4))).And(t25.eq(3).Or(t25.eq(2).And(t23.neq(2)))).unmask(0)
radd = (ee.ImageCollection('projects/radar-wur/raddalert/v1').filterMetadata('layer', 'contains', 'alert')
        .filterMetadata('geography', 'equals', 'asia').sort('system:time_end', False).first())
rd = radd.select('Date'); f_radd = rd.gte(24001).And(rd.lte(25366)).unmask(0)
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(AOI).select('trees')
f_dw = dw.filterDate('2023-01-01', '2024-01-01').mean().gte(0.5).And(dw.filterDate('2025-06-01', '2026-01-01').mean().lt(0.3)).unmask(0)
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
valid = gsw.lt(50).And(wc.neq(50)).And(wc.neq(80))
eco = (ee.Image(3).where(tc.gte(10), 2).where(tc.gte(50), 1).where(wc.eq(90).Or(wc.eq(95)), 4)).rename('eco').toInt()
flags = ee.Image.cat([f_hansen.rename('Hansen'), f_tmf.rename('TMF'), f_radd.rename('RADD'), f_dw.rename('DW')]).toInt()
stratum = flags.reduce(ee.Reducer.sum()).rename('strata').toInt()
STACK = flags.addBands(stratum).addBands(eco).updateMask(valid).clip(AOI)

def area_by(mask_img, group_band, scales=(60, 100, 200)):
    # Luas (ha) per kelompok; bila komputasi terlalu berat, ulangi pada skala lebih kasar
    img = ee.Image.pixelArea().divide(1e4).updateMask(mask_img.mask()).addBands(STACK.select(group_band))
    for sc in scales:
        try:
            g = img.reduceRegion(ee.Reducer.sum().group(1, group_band), AOI, sc, maxPixels=1e12, tileScale=16).get('groups').getInfo()
            print(f'  luas per {group_band} dihitung pada skala {sc} m'); return g
        except Exception as e:
            print(f'  skala {sc} m gagal ({str(e)[:80]}), coba lebih kasar...')
    raise RuntimeError('Perhitungan luas gagal pada semua skala')

if os.path.exists(F_POINTS) and os.path.exists(F_STRATA):
    pts = pd.read_csv(F_POINTS); W = json.load(open(F_STRATA)); print('Titik sudah ada:', len(pts), '(tidak diambil ulang)')
else:
    W = {int(x['strata']): x['sum'] for x in area_by(STACK.select('strata'), 'strata')}   # ha per strata
    json.dump(W, open(F_STRATA, 'w'))
    fc = STACK.stratifiedSample(numPoints=0, classBand='strata', classValues=list(ALLOC), classPoints=list(ALLOC.values()),
                                region=AOI, scale=30, seed=SEED, geometries=True, tileScale=16)
    feats = fc.getInfo()['features']
    pts = pd.DataFrame([{**f['properties'], 'lon': f['geometry']['coordinates'][0], 'lat': f['geometry']['coordinates'][1]} for f in feats])
    pts = pts.sample(frac=1, random_state=SEED).reset_index(drop=True); pts.insert(0, 'pid', range(1, len(pts) + 1))   # urutan acak (buta strata)
    pts.to_csv(F_POINTS, index=False)
print(pd.Series({int(k): round(v, 1) for k, v in W.items()}, name='luas_ha_per_strata'))
print(pts.groupby('strata').size().rename('jumlah_titik'))

In [ ]:
# A2. Luas tiap produk per ekosistem (wall-to-wall) dan sampel acak untuk kesepakatan antarproduk
if not os.path.exists(F_WALL):
    wall = {}
    for p in PRODUCTS:
        wall[p] = {int(x['eco']): x['sum'] for x in area_by(STACK.select(p).selfMask(), 'eco')}
    wall['total'] = {int(x['eco']): x['sum'] for x in area_by(STACK.select('eco'), 'eco')}
    json.dump(wall, open(F_WALL, 'w'))
if not os.path.exists(F_AGREE):
    parts = []                                   # Earth Engine membatasi 5.000 baris per getInfo -> 5 x 4.000 piksel
    for k in range(5):
        rp = STACK.select(PRODUCTS + ['eco']).sample(region=AOI, scale=30, numPixels=4000, seed=SEED + k, tileScale=16, geometries=False)
        parts.append(pd.DataFrame([f['properties'] for f in rp.getInfo()['features']])); print(f'  sampel kesepakatan {k + 1}/5: {len(parts[-1])} piksel')
    pd.concat(parts, ignore_index=True).to_csv(F_AGREE, index=False)
wall = json.load(open(F_WALL)); print({p: round(sum(v.values()), 0) for p, v in wall.items()})

In [ ]:
# A3. Potongan citra Sentinel-2 untuk setiap titik (1,5 × 1,5 km), disimpan di Drive
import requests
from concurrent.futures import ThreadPoolExecutor
cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
def s2(a, b):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI).filterDate(a, b).linkCollection(cs, ['cs_cdf'])
            .map(lambda i: i.updateMask(i.select('cs_cdf').gte(0.6))).median())
PERIODS = [('A', '2023-06-01', '2023-11-01', 'rgb'), ('B', '2024-06-01', '2024-11-01', 'rgb'), ('C', '2025-06-01', '2025-11-01', 'rgb'),
           ('D', '2025-11-01', '2026-04-01', 'rgb'), ('E', '2025-11-01', '2026-04-01', 'fc')]
COMP = {k: s2(a, b) for k, a, b, _ in PERIODS}
VIS = {'rgb': {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 1500}, 'fc': {'bands': ['B8', 'B11', 'B4'], 'min': 0, 'max': 4000}}
def chip(row):
    pt = ee.Geometry.Point([row.lon, row.lat]); reg = pt.buffer(750).bounds()
    box = ee.Image().paint(ee.FeatureCollection([ee.Feature(pt.buffer(30).bounds())]), 0, 2).visualize(palette=['ff0000'])
    out = 0
    for k, a, b, v in PERIODS:
        fn = os.path.join(CHIPS, f'{int(row.pid):03d}_{k}.png')
        if os.path.exists(fn) and os.path.getsize(fn) > 1000: continue
        img = COMP[k].visualize(**VIS[v]).blend(box)
        for attempt in range(3):
            try:
                r = requests.get(img.getThumbURL({'region': reg, 'dimensions': 220, 'format': 'png'}), timeout=120)
                if r.status_code == 200 and len(r.content) > 1000:
                    open(fn, 'wb').write(r.content); out += 1; break
            except Exception:
                time.sleep(3)
    return out
t0 = time.perf_counter(); done = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    for i, n in enumerate(ex.map(chip, pts.itertuples())):
        done += n
        if (i + 1) % 25 == 0: print(f'{i + 1}/{len(pts)} titik ({time.perf_counter() - t0:.0f} dtk)')
missing = [p for p in pts.pid for k, *_ in PERIODS if not os.path.exists(os.path.join(CHIPS, f'{int(p):03d}_{k}.png'))]
print('Selesai. Chip baru:', done, '| chip yang belum ada:', len(missing), '(jalankan ulang sel ini bila > 0)')

# Bagian B — Interpretasi visual (buta)
Untuk setiap titik, perhatikan **kotak merah di tengah** (± 60 × 60 m) pada lima potongan citra:
**A** = Jun–Okt 2023 · **B** = Jun–Okt 2024 · **C** = Jun–Okt 2025 · **D** = Nov 2025–Mar 2026 · **E** = D dalam *false color* NIR-SWIR-merah (vegetasi = jingga/merah, tanah terbuka = hijau pucat/krem).

| Tombol | Kapan dipilih |
|---|---|
| **1 · Dibuka 2024–2025** | Di A masih bervegetasi, lalu di B, C, atau D menjadi tanah terbuka, pola blok/petak, jalan baru, atau kebun baru |
| **0 · Tidak berubah** | Tutupan sama dari A sampai D (hutan tetap hutan, savana tetap savana) |
| **2 · Sudah terbuka sejak 2023** | Di A sudah berupa lahan terbuka, sawah, kebun, atau permukiman |
| **3 · Berubah alami** | Bekas terbakar atau tergenang **tanpa** tanda konversi (tidak ada pola blok/jalan) |
| **9 · Tidak dapat ditentukan** | Awan, citra kosong, atau ragu |

Jika awan menutupi satu periode, gunakan periode lain. Nilai **piksel di dalam kotak merah**, bukan sekelilingnya. Label tersimpan otomatis setiap kali tombol diklik.

In [ ]:
import ipywidgets as Wd
from IPython.display import display, clear_output
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception:
    pass
LBL_BTN = [(1, '1 · Dibuka 2024–2025', 'danger'), (0, '0 · Tidak berubah', 'success'), (2, '2 · Sudah terbuka sejak 2023', 'warning'),
           (3, '3 · Berubah alami', 'info'), (9, '9 · Tidak dapat ditentukan', '')]
TITLES = {'A': 'A · Jun–Okt 2023', 'B': 'B · Jun–Okt 2024', 'C': 'C · Jun–Okt 2025', 'D': 'D · Nov 2025–Mar 2026', 'E': 'E · D false color'}

def run_app(order, fpath, title):
    labs = pd.read_csv(fpath) if os.path.exists(fpath) else pd.DataFrame(columns=['pid', 'label', 'waktu'])
    state = {'i': 0}
    done = set(labs.pid.astype(int))
    todo = [p for p in order if p not in done]
    head = Wd.HTML(); imgs = Wd.HBox(); msg = Wd.HTML()
    def show():
        if state['i'] >= len(todo):
            head.value = f'<h3>{title}: selesai ✔ ({len(done)} titik)</h3>'; imgs.children = []; return
        p = todo[state['i']]
        head.value = f'<h3>{title} — titik {p} &nbsp; ({len(done)}/{len(order)} selesai)</h3>'
        boxes = []
        for k in 'ABCDE':
            fn = os.path.join(CHIPS, f'{int(p):03d}_{k}.png')
            im = Wd.Image(value=open(fn, 'rb').read() if os.path.exists(fn) else b'', format='png', width=230, height=230)
            boxes.append(Wd.VBox([Wd.HTML(f'<b>{TITLES[k]}</b>'), im]))
        imgs.children = boxes
    def click(val):
        def f(_):
            nonlocal labs
            if state['i'] >= len(todo): return
            p = todo[state['i']]
            labs = pd.concat([labs[labs.pid != p], pd.DataFrame([{'pid': p, 'label': val, 'waktu': time.strftime('%Y-%m-%d %H:%M:%S')}])])
            labs.to_csv(fpath, index=False); done.add(p)
            msg.value = f'Titik {p} = <b>{val}</b> tersimpan.'; state['i'] += 1; show()
        return f
    def back(_):
        if state['i'] > 0:
            state['i'] -= 1; p = todo[state['i']]; done.discard(p); msg.value = f'Kembali ke titik {p}. Pilih ulang labelnya.'; show()
    btns = [Wd.Button(description=d, button_style=s, layout=Wd.Layout(width='230px')) for _, d, s in LBL_BTN]
    for b, (v, _, _) in zip(btns, LBL_BTN): b.on_click(click(v))
    bk = Wd.Button(description='↩ Kembali', layout=Wd.Layout(width='120px')); bk.on_click(back)
    display(Wd.VBox([head, imgs, Wd.HBox(btns + [bk]), msg])); show()

ORDER = pts.pid.astype(int).tolist()          # urutan sudah diacak; strata dan hasil produk tidak ditampilkan
run_app(ORDER, F_LABELS, 'Interpretasi')

### B2. Uji konsistensi (setelah 300 titik selesai)
30 titik acak ditampilkan ulang dalam urutan berbeda. Nilai lagi **tanpa melihat** label sebelumnya. Hasilnya dipakai untuk melaporkan konsistensi interpreter.

In [ ]:
n_lab = len(pd.read_csv(F_LABELS)) if os.path.exists(F_LABELS) else 0
if n_lab < len(pts):
    print(f'Belum waktunya: baru {n_lab} dari {len(pts)} titik dinilai. Kembali ke aplikasi di sel atas, sel ini dijalankan setelah 300 titik selesai.')
else:
    labs = pd.read_csv(F_LABELS)
    rng = np.random.default_rng(7); RE_ORDER = rng.choice(labs.pid.values, 30, replace=False).tolist()
    run_app(RE_ORDER, F_RELAB, 'Uji konsistensi')

# Bagian C — Analisis akurasi (estimator berstrata) dan bias per ekosistem

In [ ]:
from sklearn.metrics import cohen_kappa_score
labs = pd.read_csv(F_LABELS); D = pts.merge(labs[['pid', 'label']], on='pid', how='inner')
counts = D.label.value_counts().sort_index()
lab_tab = pd.DataFrame({'kelas_interpretasi': ['1 Dibuka 2024–2025', '0 Tidak berubah', '2 Sudah terbuka sejak 2023', '3 Berubah alami', '9 Tidak dapat ditentukan'],
                        'kode': [1, 0, 2, 3, 9]})
lab_tab['jumlah'] = lab_tab.kode.map(counts).fillna(0).astype(int); lab_tab['persen'] = (100 * lab_tab.jumlah / len(D)).round(1)
save_tab(lab_tab.drop(columns='kode'), 'hasil_interpretasi', 'interpretation_results')
D = D[D.label != 9].copy(); D['ref'] = (D.label == 1).astype(int)
Wh = {int(k): v for k, v in W.items()}; A_TOT = sum(Wh.values())
nh = D.groupby('strata').size().to_dict()

D = D.reset_index(drop=True)
_wpt = D.strata.map(lambda h: Wh[h] / nh[h]).values                     # bobot per titik = luas strata / jumlah titik strata
_groups = [np.where(D.strata.values == h)[0] for h in sorted(nh)]
_rng = np.random.default_rng(SEED)
BOOT = [np.concatenate([g[_rng.integers(0, len(g), len(g))] for g in _groups]) for _ in range(2000)]   # indeks bootstrap dalam strata

def est(d, flag=None, dom=None, B=None):
    # Estimator berstrata (Stehman 2014) dengan CI 95% bootstrap dalam strata (indeks yang sama untuk semua produk)
    r_all, e_all = d.ref.values, d.eco.values; f_all = d[flag].values if flag else None
    def one(ix):
        w, r = _wpt[ix], r_all[ix]; m = np.ones(len(ix), bool) if dom is None else (e_all[ix] == dom)
        w, r = w[m], r[m]; o = {'p_ref': np.sum(w * r) / np.sum(w) if len(w) else np.nan}
        if flag:
            f = f_all[ix][m]; tp = np.sum(w * (f & r))
            o['OA'] = np.sum(w * (f == r)) / np.sum(w) if len(w) else np.nan
            o['UA'] = tp / np.sum(w * f) if np.sum(w * f) > 0 else np.nan
            o['PA'] = tp / np.sum(w * r) if np.sum(w * r) > 0 else np.nan
            o['F1'] = 2 * o['UA'] * o['PA'] / (o['UA'] + o['PA']) if (o['UA'] + o['PA']) > 0 else np.nan
        return o
    base = one(np.arange(len(d))); bt = pd.DataFrame([one(ix) for ix in (BOOT if B is None else BOOT[:B])])
    return {k: (v, np.nanpercentile(bt[k], 2.5), np.nanpercentile(bt[k], 97.5)) for k, v in base.items()}

rows = []
for p in PRODUCTS:
    e = est(D, p)
    for met, nm in (('OA', 'akurasi_keseluruhan'), ('UA', 'akurasi_pengguna'), ('PA', 'akurasi_produser'), ('F1', 'F1')):
        rows.append({'produk': p, 'metrik': nm, 'nilai': round(e[met][0], 4), 'batas_bawah': round(e[met][1], 4), 'batas_atas': round(e[met][2], 4)})
acc = pd.DataFrame(rows)
acc_w = acc.pivot(index='produk', columns='metrik', values='nilai').loc[PRODUCTS].reset_index()
save_tab(acc, 'akurasi_produk_dengan_CI', 'product_accuracy_with_CI'); save_tab(acc_w, 'akurasi_produk', 'product_accuracy')
print(acc_w.to_string(index=False))

# Luas pembukaan: estimasi dari referensi vs luas peta tiap produk
e0 = est(D)
area = [{'produk': 'Referensi (interpretasi)', 'luas_ha': round(e0['p_ref'][0] * A_TOT, 0), 'batas_bawah': round(e0['p_ref'][1] * A_TOT, 0), 'batas_atas': round(e0['p_ref'][2] * A_TOT, 0)}]
for p in PRODUCTS:
    area.append({'produk': p, 'luas_ha': round(sum(wall[p].values()), 0), 'batas_bawah': np.nan, 'batas_atas': np.nan})
area = pd.DataFrame(area); save_tab(area, 'luas_pembukaan', 'cleared_area'); print(area.to_string(index=False))

# Akurasi produser (omisi) dan pengguna per ekosistem
erow = []
for k, nm in ECO.items():
    sub = D[D.eco == k]
    for p in PRODUCTS:
        e = est(D, p, dom=k, B=1000)
        erow.append({'ekosistem': nm, 'produk': p, 'n_titik_ekosistem': len(sub), 'n_referensi_dibuka': int(sub.ref.sum()),
                     'akurasi_produser': round(e['PA'][0], 4), 'PA_bawah': round(e['PA'][1], 4), 'PA_atas': round(e['PA'][2], 4),
                     'akurasi_pengguna': round(e['UA'][0], 4)})
ecot = pd.DataFrame(erow); save_tab(ecot, 'akurasi_per_ekosistem', 'accuracy_by_ecosystem')
ecoarea = pd.DataFrame([{'ekosistem': ECO[int(k)], 'luas_ha': round(v, 0), **{f'luas_peta_{p}_ha': round(wall[p].get(k, wall[p].get(str(k), 0)), 0) for p in PRODUCTS}}
                        for k, v in wall['total'].items()])
save_tab(ecoarea, 'luas_ekosistem_dan_produk', 'ecosystem_and_product_area')
print(ecot.pivot(index='ekosistem', columns='produk', values='akurasi_produser').round(3))

# Kesepakatan antarproduk (20.000 piksel acak)
ag = pd.read_csv(F_AGREE).dropna(subset=PRODUCTS); kp = []
for i, a in enumerate(PRODUCTS):
    for b in PRODUCTS[i + 1:]:
        kp.append({'produk_A': a, 'produk_B': b, 'kappa': round(cohen_kappa_score(ag[a], ag[b]), 4), 'kesepakatan': round((ag[a] == ag[b]).mean(), 4)})
kp = pd.DataFrame(kp); save_tab(kp, 'kesepakatan_antarproduk', 'inter_product_agreement'); print(kp.to_string(index=False))

# Strata dan alokasi
st = pd.DataFrame([{'strata': h, 'luas_ha': round(Wh[h], 0), 'bobot': round(Wh[h] / A_TOT, 5), 'jumlah_titik': nh.get(h, 0)} for h in sorted(Wh)])
save_tab(st, 'strata_dan_alokasi', 'strata_and_allocation')

# Konsistensi interpreter
if os.path.exists(F_RELAB):
    r2 = pd.read_csv(F_RELAB).merge(pd.read_csv(F_LABELS), on='pid', suffixes=('_ulang', '_awal'))
    cons = {'n': len(r2), 'kesepakatan_5kelas': round(float((r2.label_ulang == r2.label_awal).mean()), 4),
            'kesepakatan_biner': round(float(((r2.label_ulang == 1) == (r2.label_awal == 1)).mean()), 4),
            'kappa_biner': round(float(cohen_kappa_score(r2.label_ulang == 1, r2.label_awal == 1)), 4)}
else:
    cons = {'n': 0}
print('Konsistensi interpreter:', cons)
D.to_csv(os.path.join(DIR_DATA, 'T04_titik_berlabel.csv'), index=False)

In [ ]:
# Gambar dua bahasa
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
COL = {'Hansen': '#4C78A8', 'TMF': '#F58518', 'RADD': '#54A24B', 'DW': '#B279A2'}
L = {'ID': dict(ua='Akurasi pengguna', pa='Akurasi produser', f1='F1', area='Luas pembukaan 2024–2025 (ha)', ref='Referensi\n(interpretasi)',
                eco='Ekosistem', prod='Produk', kap='Kappa Cohen', ecos=ECO, case=['Omisi (produk: tidak ada; referensi: dibuka)', 'Komisi (produk: ada; referensi: tidak dibuka)']),
     'EN': dict(ua="User's accuracy", pa="Producer's accuracy", f1='F1', area='Cleared area 2024–2025 (ha)', ref='Reference\n(interpretation)',
                eco='Ecosystem', prod='Product', kap="Cohen's kappa", ecos=ECO_EN, case=['Omission (product: no; reference: cleared)', 'Commission (product: yes; reference: not cleared)'])}
for lang, t in L.items():
    # 1. UA, PA, F1 per produk + CI
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
    for a, m, nm in zip(ax, ['akurasi_pengguna', 'akurasi_produser', 'F1'], [t['ua'], t['pa'], t['f1']]):
        d = acc[acc.metrik == m].set_index('produk').loc[PRODUCTS]
        a.bar(range(4), d.nilai, color=[COL[p] for p in PRODUCTS], yerr=[d.nilai - d.batas_bawah, d.batas_atas - d.nilai], capsize=4)
        a.set_xticks(range(4)); a.set_xticklabels([TR[lang][p] for p in PRODUCTS], rotation=20); a.set_title(nm); a.set_ylim(0, 1.05)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_akurasi_produk_{lang}.png'), dpi=300); plt.close(fig)
    # 2. PA per ekosistem (heatmap)
    hm = ecot.pivot(index='ekosistem', columns='produk', values='akurasi_produser').reindex(list(ECO.values()))[PRODUCTS]
    hm.index = [t['ecos'][k] for k in ECO]; hm.columns = [TR[lang][p] for p in PRODUCTS]
    fig, ax = plt.subplots(figsize=(7, 3.4)); sns.heatmap(hm, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, cbar_kws={'label': t['pa']}, ax=ax)
    ax.set_xlabel(t['prod']); ax.set_ylabel(t['eco']); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_PA_per_ekosistem_{lang}.png'), dpi=300); plt.close(fig)
    # 3. Luas
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    names = [t['ref']] + [TR[lang][p] for p in PRODUCTS]; vals = area.luas_ha.values
    ax.bar(range(5), vals, color=['#333333'] + [COL[p] for p in PRODUCTS],
           yerr=[[vals[0] - area.batas_bawah[0]] + [0] * 4, [area.batas_atas[0] - vals[0]] + [0] * 4], capsize=4)
    ax.set_xticks(range(5)); ax.set_xticklabels(names, rotation=15); ax.set_ylabel(t['area'])
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_luas_pembukaan_{lang}.png'), dpi=300); plt.close(fig)
    # 4. Kappa antarproduk
    K = pd.DataFrame(np.eye(4), index=PRODUCTS, columns=PRODUCTS)
    for r in kp.itertuples(): K.loc[r.produk_A, r.produk_B] = K.loc[r.produk_B, r.produk_A] = r.kappa
    K.index = K.columns = [TR[lang][p] for p in PRODUCTS]
    fig, ax = plt.subplots(figsize=(5, 4)); sns.heatmap(K, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': t['kap']}, ax=ax)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_kappa_antarproduk_{lang}.png'), dpi=300); plt.close(fig)
    # 5. Contoh kasus omisi dan komisi Hansen (chip A dan D)
    om = D[(D.ref == 1) & (D.Hansen == 0)].head(2); cm = D[(D.ref == 0) & (D.Hansen == 1)].head(2)
    cases = [(r, t['case'][0]) for r in om.itertuples()] + [(r, t['case'][1]) for r in cm.itertuples()]
    if cases:
        fig, ax = plt.subplots(len(cases), 2, figsize=(5.2, 2.7 * len(cases)), squeeze=False)
        for i, (r, ttl) in enumerate(cases):
            for j, k in enumerate('AD'):
                fn = os.path.join(CHIPS, f'{int(r.pid):03d}_{k}.png')
                if os.path.exists(fn): ax[i, j].imshow(Image.open(fn))
                ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
                ax[i, j].set_title(('2023' if k == 'A' else '2025–26') + (f' — {t["ecos"][int(r.eco)]}' if j == 0 else ''), fontsize=8)
            ax[i, 0].set_ylabel(ttl.split(' (')[0], fontsize=8)
        fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_contoh_kasus_{lang}.png'), dpi=250); plt.close(fig)

# Peta strata (thumbnail)
import requests
pal = ['#f0f0f0', '#fee5d9', '#fcae91', '#fb6a4a', '#cb181d']
url = STACK.select('strata').visualize(min=0, max=4, palette=pal).getThumbURL({'region': AOI.bounds(), 'dimensions': 1400, 'format': 'png'})
open(os.path.join(DIR_FIG, 'T04_peta_strata.png'), 'wb').write(requests.get(url, timeout=300).content)
for lang, lg in (('ID', 'Jumlah produk yang menandai pembukaan 2024–2025'), ('EN', 'Number of products flagging 2024–2025 clearing')):
    fig, ax = plt.subplots(figsize=(8, 6)); ax.imshow(Image.open(os.path.join(DIR_FIG, 'T04_peta_strata.png'))); ax.axis('off')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=c, label=str(i)) for i, c in enumerate(pal)], title=lg, loc='lower left', fontsize=8, title_fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T04_gambar_peta_strata_{lang}.png'), dpi=200); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
# Ringkasan dan unduh
summary = {'jumlah_titik': int(len(pts)), 'titik_valid_setelah_buang_9': int(len(D)), 'hasil_interpretasi': lab_tab[['kelas_interpretasi', 'jumlah']].to_dict('records'),
           'luas_strata_ha': {str(k): round(v, 0) for k, v in Wh.items()}, 'akurasi_produk': acc.to_dict('records'), 'luas_pembukaan': area.to_dict('records'),
           'PA_per_ekosistem': ecot.to_dict('records'), 'kappa_antarproduk': kp.to_dict('records'), 'konsistensi_interpreter': cons}
json.dump(summary, open(os.path.join(DIR_TAB, 'T04_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=float)
import shutil
shutil.copy(F_POINTS, DIR_DATA); shutil.copy(F_LABELS, DIR_DATA)
if os.path.exists(F_RELAB): shutil.copy(F_RELAB, DIR_DATA)
zpath = '/content/T04_hasil.zip' if os.path.exists('/content') else 'T04_hasil.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for fn in fs: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
print(json.dumps({k: summary[k] for k in ['jumlah_titik', 'titik_valid_setelah_buang_9', 'konsistensi_interpreter']}, indent=1))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)

Selesai. Upload `T04_hasil.zip` ke Claude.